In [113]:
import pandas as pd
from pathlib import Path
import gspread
from gspread_dataframe import set_with_dataframe
from google.oauth2.service_account import Credentials
from gspread_formatting import *

In [114]:
DIR_PATH = Path("Consolidated_Schedule_of_Investments/2026-03-31/")
CONSOLIDATED_CSV_PATH = Path(DIR_PATH, "outputs/gecc-20260331.csv")
ANALYSIS_PATH = Path(DIR_PATH, "analysis")
ANALYSIS_PATH.mkdir(exist_ok=True)

DIR_PATH.exists(), CONSOLIDATED_CSV_PATH.exists(), ANALYSIS_PATH.exists()

(True, True, True)

In [115]:
df_investment = pd.read_csv(CONSOLIDATED_CSV_PATH)
df_investment.info()

<class 'pandas.DataFrame'>
RangeIndex: 83 entries, 0 to 82
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   portfolio_company    83 non-null     str  
 1   industry             83 non-null     str  
 2   security             83 non-null     str  
 3   interest_rate        55 non-null     str  
 4   investment_type      83 non-null     str  
 5   acquisition_date     82 non-null     str  
 6   percentage_of_class  13 non-null     str  
 7   maturity             63 non-null     str  
 8   amount_quantity      83 non-null     int64
 9   cost                 83 non-null     int64
 10  fair_value           83 non-null     int64
 11  part                 83 non-null     str  
 12  table_index          83 non-null     int64
 13  row_index            83 non-null     int64
dtypes: int64(5), str(9)
memory usage: 9.2 KB


In [116]:
# df_investment[df_investment["issuer"].str.contains("Skydio, Inc", na=False)]["cost"]

In [117]:
df_investment.head(275)

,portfolio_company,industry,security,interest_rate,investment_type,acquisition_date,percentage_of_class,maturity,amount_quantity,cost,fair_value,part,table_index,row_index
0,"ACTIV8 Health, LLC",Insurance,"1st Lien, Secured Loan",3M SOFR + 5.75% (9.41%),investments at fair value,02/03/2026,NaN,02/03/2031,3696,3660,3655,gecc-20260331,0,3
1,Advancion,Chemicals,"1st Lien, Secured Loan",3M SOFR + 4.00% (7.77%),investments at fair value,08/26/2025,NaN,11/24/2027,2974,2944,2837,gecc-20260331,0,4
2,American Coastal Insurance Corp.,Insurance,Unsecured Bond,7.25%,investments at fair value,12/20/2022,NaN,12/15/2027,8000,6046,7945,gecc-20260331,0,5
3,Auction.com,Financial Services,"1st Lien, Secured Loan",6M SOFR + 6.00% (9.63%),investments at fair value,09/09/2024,NaN,05/26/2028,1995,1921,1661,gecc-20260331,0,6
4,"CLO Formation JV, LLC",Structured Finance,Common Equity,NaN,investments at fair value,04/23/2024,71.25%,NaN,166,52358,37590,gecc-20260331,0,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
78,"Vivos Holdings, LLC",Consumer Products,Promissory Note,9.00% PIK,investments at fair value,08/13/2025,NaN,08/13/2032,94,129,94,gecc-20260331,2,27
79,"Vivos Holdings, LLC",Consumer Products,Warrants,NaN,investments at fair value,08/13/2025,*,NaN,592,0,81,gecc-20260331,2,28
80,"W&T Offshore, Inc.",Oil & Gas Exploration & Production,"2nd Lien, Secured Bond",10.75%,investments at fair value,01/14/2025,NaN,02/02/2029,1900,1632,1945,gecc-20260331,2,29
81,"Walor North America, Inc",Industrial,"1st Lien, Secured Loan",1M SOFR + 5.75% (9.55%),investments at fair value,06/17/2025,NaN,06/17/2028,1369,1369,1363,gecc-20260331,2,30


In [118]:
# convert cost 
for col in ["cost", "fair_value"]:
    df_investment[col] = (
        df_investment[col]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.replace(r"^\((.*)\)$", r"-\1", regex=True)
        .replace("nan", pd.NA)
    )

    df_investment[col] = pd.to_numeric(df_investment[col], errors="coerce").astype("Int64")

df_investment.drop(columns=["part", "table_index", "row_index"], inplace=True)
df_investment

,portfolio_company,industry,security,interest_rate,investment_type,acquisition_date,percentage_of_class,maturity,amount_quantity,cost,fair_value
0,"ACTIV8 Health, LLC",Insurance,"1st Lien, Secured Loan",3M SOFR + 5.75% (9.41%),investments at fair value,02/03/2026,NaN,02/03/2031,3696,3660,3655
1,Advancion,Chemicals,"1st Lien, Secured Loan",3M SOFR + 4.00% (7.77%),investments at fair value,08/26/2025,NaN,11/24/2027,2974,2944,2837
2,American Coastal Insurance Corp.,Insurance,Unsecured Bond,7.25%,investments at fair value,12/20/2022,NaN,12/15/2027,8000,6046,7945
3,Auction.com,Financial Services,"1st Lien, Secured Loan",6M SOFR + 6.00% (9.63%),investments at fair value,09/09/2024,NaN,05/26/2028,1995,1921,1661
4,"CLO Formation JV, LLC",Structured Finance,Common Equity,NaN,investments at fair value,04/23/2024,71.25%,NaN,166,52358,37590
...,...,...,...,...,...,...,...,...,...,...,...
78,"Vivos Holdings, LLC",Consumer Products,Promissory Note,9.00% PIK,investments at fair value,08/13/2025,NaN,08/13/2032,94,129,94
79,"Vivos Holdings, LLC",Consumer Products,Warrants,NaN,investments at fair value,08/13/2025,*,NaN,592,0,81
80,"W&T Offshore, Inc.",Oil & Gas Exploration & Production,"2nd Lien, Secured Bond",10.75%,investments at fair value,01/14/2025,NaN,02/02/2029,1900,1632,1945
81,"Walor North America, Inc",Industrial,"1st Lien, Secured Loan",1M SOFR + 5.75% (9.55%),investments at fair value,06/17/2025,NaN,06/17/2028,1369,1369,1363


In [119]:
df_investment["profit"] = df_investment["fair_value"] - df_investment["cost"]
df_investment["is_profit"] = df_investment["profit"] > 0
df_investment["is_loss"] = df_investment["profit"] < 0
df_investment["is_breakeven"] = df_investment["profit"] == 0
df_investment['Profit_Loss'] = df_investment['fair_value'] - df_investment['cost']
df_investment

,portfolio_company,industry,security,interest_rate,investment_type,acquisition_date,percentage_of_class,maturity,amount_quantity,cost,fair_value,profit,is_profit,is_loss,is_breakeven,Profit_Loss
0,"ACTIV8 Health, LLC",Insurance,"1st Lien, Secured Loan",3M SOFR + 5.75% (9.41%),investments at fair value,02/03/2026,NaN,02/03/2031,3696,3660,3655,-5,False,True,False,-5
1,Advancion,Chemicals,"1st Lien, Secured Loan",3M SOFR + 4.00% (7.77%),investments at fair value,08/26/2025,NaN,11/24/2027,2974,2944,2837,-107,False,True,False,-107
2,American Coastal Insurance Corp.,Insurance,Unsecured Bond,7.25%,investments at fair value,12/20/2022,NaN,12/15/2027,8000,6046,7945,1899,True,False,False,1899
3,Auction.com,Financial Services,"1st Lien, Secured Loan",6M SOFR + 6.00% (9.63%),investments at fair value,09/09/2024,NaN,05/26/2028,1995,1921,1661,-260,False,True,False,-260
4,"CLO Formation JV, LLC",Structured Finance,Common Equity,NaN,investments at fair value,04/23/2024,71.25%,NaN,166,52358,37590,-14768,False,True,False,-14768
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
78,"Vivos Holdings, LLC",Consumer Products,Promissory Note,9.00% PIK,investments at fair value,08/13/2025,NaN,08/13/2032,94,129,94,-35,False,True,False,-35
79,"Vivos Holdings, LLC",Consumer Products,Warrants,NaN,investments at fair value,08/13/2025,*,NaN,592,0,81,81,True,False,False,81
80,"W&T Offshore, Inc.",Oil & Gas Exploration & Production,"2nd Lien, Secured Bond",10.75%,investments at fair value,01/14/2025,NaN,02/02/2029,1900,1632,1945,313,True,False,False,313
81,"Walor North America, Inc",Industrial,"1st Lien, Secured Loan",1M SOFR + 5.75% (9.55%),investments at fair value,06/17/2025,NaN,06/17/2028,1369,1369,1363,-6,False,True,False,-6


In [120]:
df_summary = pd.DataFrame({
    "a": [
        "Total Investment Count",
        "Total Cost", 
        "Total FV", 
        "P&L"
    ],
    "b": [
        len(df_investment),
        int(df_investment["cost"].sum()),
        int(df_investment["fair_value"].sum()),
        int(df_investment["fair_value"].sum())-int(df_investment["cost"].sum()),
    ]
})

df_summary

,a,b
0,Total Investment Count,83
1,Total Cost,348231
2,Total FV,276763
3,P&L,-71468


In [121]:
# instrument-Level Analysis
df_instrument_level_analysis = (
    df_investment.groupby("security")
      .agg(
          total_investments=("portfolio_company", "count"),

          # counts
          profitable_count=("is_profit", "sum"),
          loss_count=("is_loss", "sum"),
          breakeven_count=("is_breakeven", "sum"),

          # financials
          total_cost=("cost", "sum"),
          total_fair_value=("fair_value", "sum"),
          total_profit=("profit", "sum"),

          # averages
          avg_profit=("profit", "mean"),

          # risk
          max_profit=("profit", "max"),
          max_loss=("profit", "min")
      )
)

df_instrument_level_analysis.reset_index(names="security", inplace=True)
df_instrument_level_analysis["avg_profit"] = round(df_instrument_level_analysis["avg_profit"], 2)

# win, breakeven, and loss rate
df_instrument_level_analysis["win_rate"] = round(
    df_instrument_level_analysis["profitable_count"] /
    df_instrument_level_analysis["total_investments"]
, 2)

df_instrument_level_analysis["breakeven_rate"] = round(
    df_instrument_level_analysis["breakeven_count"] /
    df_instrument_level_analysis["total_investments"]
, 2)

df_instrument_level_analysis["loss_rate"] = round(
    df_instrument_level_analysis["loss_count"] /
    df_instrument_level_analysis["total_investments"]
, 2)

# sort by total no investments
df_instrument_level_analysis = df_instrument_level_analysis.sort_values(
    by="total_fair_value", ascending=False
)


column_mapping = {
    "security": "Security",
    "total_investments": "Total Investments Count",
    "total_cost": "Total Security Cost",
    "total_fair_value": "Total Security FV",
    "total_profit": "Total Security P&L",
    "profitable_count": "Profitable Security Count",
    "win_rate": "Profitable %",
    "breakeven_count": "Breakeven Security Count",
    "breakeven_rate": "Breakeven %",
    "loss_count": "Loss Security Count",
    "loss_rate": "Loss %"
}
desired_order = [
    "Security",
    "Total Investments Count",
    "Total Security Cost",
    "Total Security FV",
    "Total Security P&L",
    "Profitable Security Count",
    "Profitable %",
    "Breakeven Security Count",
    "Breakeven %",
    "Loss Security Count",
    "Loss %"
]

df_instrument_level_analysis = df_instrument_level_analysis.rename(columns=column_mapping).reindex(columns=desired_order)

display(df_instrument_level_analysis)

,Security,Total Investments Count,Total Security Cost,Total Security FV,Total Security P&L,Profitable Security Count,Profitable %,Breakeven Security Count,Breakeven %,Loss Security Count,Loss %
1,"1st Lien, Secured Loan",47,146287,127410,-18877,6,0.13,3,0.06,38,0.81
5,Common Equity,8,93419,56161,-37258,0,0.0,2,0.25,6,0.75
12,Subordinated Note,1,25325,25325,0,0,0.0,1,1.0,0,0.0
3,"2nd Lien, Secured Loan",4,31878,11808,-20070,2,0.5,0,0.0,2,0.5
15,Unsecured Bond,3,9567,11374,1807,1,0.33,0,0.0,2,0.67
7,Money Market,1,9600,9600,0,0,0.0,1,1.0,0,0.0
0,"1st Lien, Secured Bond",3,7847,7796,-51,1,0.33,0,0.0,2,0.67
8,Preference Shares,1,5000,6975,1975,1,1.0,0,0.0,0,0.0
9,Private Fund,2,4311,6496,2185,2,1.0,0,0.0,0,0.0
4,CLO Equity,1,6190,6482,292,1,1.0,0,0.0,0,0.0


In [122]:
# company-Level Analysis
df_company_level_analysis = (
    df_investment.groupby("portfolio_company")
      .agg(
          total_investments=("portfolio_company", "count"),

          # counts
          profitable_count=("is_profit", "sum"),
          loss_count=("is_loss", "sum"),
          breakeven_count=("is_breakeven", "sum"),

          # financials
          total_cost=("cost", "sum"),
          total_fair_value=("fair_value", "sum"),
          total_profit=("profit", "sum"),

          # averages
          avg_profit=("profit", "mean"),

          # risk
          max_profit=("profit", "max"),
          max_loss=("profit", "min")
      )
)

df_company_level_analysis.reset_index(names="portfolio_company", inplace=True)
df_company_level_analysis["avg_profit"] = round(df_company_level_analysis["avg_profit"], 2)

# win, breakeven, and loss rate
df_company_level_analysis["win_rate"] = round(
    df_company_level_analysis["profitable_count"] /
    df_company_level_analysis["total_investments"]
, 2)

df_company_level_analysis["breakeven_rate"] = round(
    df_company_level_analysis["breakeven_count"] /
    df_company_level_analysis["total_investments"]
, 2)

df_company_level_analysis["loss_rate"] = round(
    df_company_level_analysis["loss_count"] /
    df_company_level_analysis["total_investments"]
, 2)

# sort by total no investments
df_company_level_analysis = df_company_level_analysis.sort_values(
    by="total_fair_value", ascending=False
)


column_mapping = {
    "portfolio_company": "Portfolio Company",
    "total_investments": "Total Investments Count",
    "total_cost": "Total Company Cost",
    "total_fair_value": "Total Company FV",
    "total_profit": "Total Company P&L",
    "profitable_count": "Profitable Company Count",
    "win_rate": "Profitable %",
    "breakeven_count": "Breakeven Company Count",
    "breakeven_rate": "Breakeven %",
    "loss_count": "Loss Company Count",
    "loss_rate": "Loss %"
}
desired_order = [
    "Portfolio Company",
    "Total Investments Count",
    "Total Company Cost",
    "Total Company FV",
    "Total Company P&L",
    "Profitable Company Count",
    "Profitable %",
    "Breakeven Company Count",
    "Breakeven %",
    "Loss Company Count",
    "Loss %"
]

df_company_level_analysis = df_company_level_analysis.rename(columns=column_mapping).reindex(columns=desired_order)

display(df_company_level_analysis)

,Portfolio Company,Total Investments Count,Total Company Cost,Total Company FV,Total Company P&L,Profitable Company Count,Profitable %,Breakeven Company Count,Breakeven %,Loss Company Count,Loss %
25,"Great Elm Specialty Finance, LLC",2,42325,37790,-4535,0,0.0,1,0.5,1,0.5
4,"CLO Formation JV, LLC",1,52358,37590,-14768,0,0.0,0,0.0,1,1.0
56,"Vivos Holdings, LLC",5,17253,17398,145,2,0.4,1,0.2,2,0.4
10,"Coreweave Compute Acquisition Co. II, LLC",1,10912,11157,245,1,1.0,0,0.0,0,0.0
31,MFB Northern Inst Funds Treas Portfolio Premie...,1,9600,9600,0,0,0.0,1,1.0,0,0.0
52,Universal Fiber Systems,4,12829,8168,-4661,1,0.25,2,0.5,1,0.25
2,American Coastal Insurance Corp.,1,6046,7945,1899,1,1.0,0,0.0,0,0.0
36,"NY Daily News Enterprises, LLC",1,7000,7000,0,0,0.0,1,1.0,0,0.0
50,Trouvaille Re Ltd.,1,5000,6975,1975,1,1.0,0,0.0,0,0.0
14,Dorel Industries Inc.,2,6555,6592,37,1,0.5,0,0.0,1,0.5


In [123]:
# investments_type-Level Analysis
df_investmentsType_level_analysis = (
    df_investment.groupby("investment_type")
      .agg(
          total_investments=("investment_type", "count"),

          # counts
          profitable_count=("is_profit", "sum"),
          loss_count=("is_loss", "sum"),
          breakeven_count=("is_breakeven", "sum"),

          # financials
          total_cost=("cost", "sum"),
          total_fair_value=("fair_value", "sum"),
          total_profit=("profit", "sum"),

          # averages
          avg_profit=("profit", "mean"),

          # risk
          max_profit=("profit", "max"),
          max_loss=("profit", "min")
      )
)

df_investmentsType_level_analysis.reset_index(names="investment_type", inplace=True)
df_investmentsType_level_analysis["avg_profit"] = round(df_investmentsType_level_analysis["avg_profit"], 2)

# win, breakeven, and loss rate
df_investmentsType_level_analysis["win_rate"] = round(
    df_investmentsType_level_analysis["profitable_count"] /
    df_investmentsType_level_analysis["total_investments"]
, 2)

df_investmentsType_level_analysis["breakeven_rate"] = round(
    df_investmentsType_level_analysis["breakeven_count"] /
    df_investmentsType_level_analysis["total_investments"]
, 2)

df_investmentsType_level_analysis["loss_rate"] = round(
    df_investmentsType_level_analysis["loss_count"] /
    df_investmentsType_level_analysis["total_investments"]
, 2)

# sort by total no investments
df_investmentsType_level_analysis = df_investmentsType_level_analysis.sort_values(
    by="total_fair_value", ascending=False
)


column_mapping = {
    "investment_type":"Investment Type",
    "total_investments": "Total Investment Count",
    "total_cost": "Total Investment Type Cost",
    "total_fair_value": "Total Investment Type FV",
    "total_profit": "Total Investment Type P&L",
    "avg_profit": "Avg Profit/Investment Type",
    "max_profit": "Max Profit Investment Type",
    "max_loss": "Max Loss Investment Type",
    "profitable_count": "Profitable Investment Type Count",
    "win_rate": "Profitable %",
    "breakeven_count": "Breakeven Investment Type Count",
    "breakeven_rate": "Breakeven %",
    "loss_count": "Loss Investment Type Count",
    "loss_rate": "Loss %"
}
desired_order = [
    "Investment Type",
    "Total Investment Count",
    "Total Investment Type Cost",
    "Total Investment Type FV",
    "Total Investment Type P&L",
    "Profitable Investment Type Count",
    "Profitable %",
    "Breakeven Investment Type Count",
    "Breakeven %",
    "Loss Investment Type Count",
    "Loss %"
]

df_investmentsType_level_analysis = df_investmentsType_level_analysis.rename(columns=column_mapping).reindex(columns=desired_order)

display(df_investmentsType_level_analysis)

,Investment Type,Total Investment Count,Total Investment Type Cost,Total Investment Type FV,Total Investment Type P&L,Profitable Investment Type Count,Profitable %,Breakeven Investment Type Count,Breakeven %,Loss Investment Type Count,Loss %
0,investments at fair value,82,338631,267163,-71468,21,0.26,8,0.1,53,0.65
1,short-term investments,1,9600,9600,0,0,0.0,1,1.0,0,0.0


In [124]:
# Industry-Level Analysis
df_industry_level_analysis = (
    df_investment.groupby("industry")
      .agg(
          total_investments=("portfolio_company", "count"),

          # counts
          profitable_count=("is_profit", "sum"),
          loss_count=("is_loss", "sum"),
          breakeven_count=("is_breakeven", "sum"),

          # financials
          total_cost=("cost", "sum"),
          total_fair_value=("fair_value", "sum"),
          total_profit=("profit", "sum"),

          # averages
          avg_profit=("profit", "mean"),

          # risk
          max_profit=("profit", "max"),
          max_loss=("profit", "min")
      )
)

df_industry_level_analysis.reset_index(names="industry", inplace=True)
df_industry_level_analysis["avg_profit"] = round(df_industry_level_analysis["avg_profit"], 2)


df_industry_level_analysis["win_rate"] = round(
    df_industry_level_analysis["profitable_count"] /
    df_industry_level_analysis["total_investments"]
, 2)

df_industry_level_analysis["breakeven_rate"] = round(
    df_industry_level_analysis["breakeven_count"] /
    df_industry_level_analysis["total_investments"]
, 2)

df_industry_level_analysis["loss_rate"] = round(
    df_industry_level_analysis["loss_count"] /
    df_industry_level_analysis["total_investments"]
, 2)


df_industry_level_analysis = df_industry_level_analysis.sort_values(
    by="total_fair_value", ascending=False
)


column_mapping = {
    "total_investments": "Total Investment Count",
    "total_cost": "Total Investment Cost",
    "total_fair_value": "Total Investment FV",
    "total_profit": "Total Investment P&L",
    "avg_profit": "Avg Profit/Investment",
    "max_profit": "Max Profit Investment",
    "max_loss": "Max Loss Investment",
    "profitable_count": "Profitable Investment Count",
    "win_rate": "Profitable %",
    "breakeven_count": "Breakeven Investment Count",
    "breakeven_rate": "Breakeven %",
    "loss_count": "Loss Investment Count",
    "loss_rate": "Loss %",
    "industry": "Industry"
}
desired_order = [
    "Industry",
    "Total Investment Count",
    "Total Investment Cost",
    "Total Investment FV",
    "Total Investment P&L",
    "Profitable Investment Count",
    "Profitable %",
    "Breakeven Investment Count",
    "Breakeven %",
    "Loss Investment Count",
    "Loss %"
]

df_industry_level_analysis = df_industry_level_analysis.rename(columns=column_mapping).reindex(columns=desired_order)

display(df_industry_level_analysis)

,Industry,Total Investment Count,Total Investment Cost,Total Investment FV,Total Investment P&L,Profitable Investment Count,Profitable %,Breakeven Investment Count,Breakeven %,Loss Investment Count,Loss %
25,Structured Finance,2,58548,44072,-14476,1,0.5,0,0.0,1,0.5
24,Specialty Finance,2,42325,37790,-4535,0,0.0,1,0.5,1,0.5
26,Technology,8,27981,30034,2053,5,0.62,0,0.0,3,0.38
4,Consumer Products,9,26218,26373,155,3,0.33,1,0.11,5,0.56
14,Insurance,3,14706,18575,3869,2,0.67,0,0.0,1,0.33
13,Industrial,4,13989,13522,-467,0,0.0,0,0.0,4,1.0
11,Food & Staples,8,29785,13289,-16496,1,0.12,1,0.12,6,0.75
3,Chemicals,6,20968,11370,-9598,1,0.17,2,0.33,3,0.5
23,Short-Term Investments,1,9600,9600,0,0,0.0,1,1.0,0,0.0
18,Metals & Mining,6,14142,8392,-5750,1,0.17,0,0.0,5,0.83


In [125]:
# asset_list = list(df_asset_class_level_analysis["Asset Class"])
# name = [v.split("—")[0].strip() for v in asset_list]
# perc = [v.split("—")[1] for v in asset_list]

# df_asset_class_level_analysis = df_asset_class_level_analysis.drop(columns=["Asset Class"])
# df_asset_class_level_analysis.insert(0, "Assets", name)
# df_asset_class_level_analysis.insert(1, "percentage", perc)

# df_asset_class_level_analysis

In [132]:
column_mapping = { 
    "portfolio_company": "Portfolio Company",
    "industry": "Industry",
    "security": "Security",
    "interest_rate": "Interest Rate",
    "investment_type": "Investment Type",
    "acquisition_date": "Initial Acquisition Date",
    "percentage_of_class": "Percentage of Class",
    "maturity": "Maturity Date",
    "Profit_Loss": "P&L",
    "amount_quantity": "Par Amount / Quantity",
    "cost": "Cost",
    "fair_value": "Fair Value"
}

desired_order = [ 
    "Portfolio Company", 
    "Industry", 
    "Security", 
    "Maturity Date",
    "Initial Acquisition Date",
    "Par Amount / Quantity",
    "Interest Rate", 
    "Investment Type",
    "Percentage of Class",
    "Cost", 
    "Fair Value",
    "P&L"
]

df_investment_formatted = df_investment.rename(columns=column_mapping).reindex(columns=desired_order)
df_investment_formatted

,Portfolio Company,Industry,Security,Maturity Date,Initial Acquisition Date,Par Amount / Quantity,Interest Rate,Investment Type,Percentage of Class,Cost,Fair Value,P&L
0,"ACTIV8 Health, LLC",Insurance,"1st Lien, Secured Loan",02/03/2031,02/03/2026,3696,3M SOFR + 5.75% (9.41%),investments at fair value,NaN,3660,3655,-5
1,Advancion,Chemicals,"1st Lien, Secured Loan",11/24/2027,08/26/2025,2974,3M SOFR + 4.00% (7.77%),investments at fair value,NaN,2944,2837,-107
2,American Coastal Insurance Corp.,Insurance,Unsecured Bond,12/15/2027,12/20/2022,8000,7.25%,investments at fair value,NaN,6046,7945,1899
3,Auction.com,Financial Services,"1st Lien, Secured Loan",05/26/2028,09/09/2024,1995,6M SOFR + 6.00% (9.63%),investments at fair value,NaN,1921,1661,-260
4,"CLO Formation JV, LLC",Structured Finance,Common Equity,NaN,04/23/2024,166,NaN,investments at fair value,71.25%,52358,37590,-14768
...,...,...,...,...,...,...,...,...,...,...,...,...
78,"Vivos Holdings, LLC",Consumer Products,Promissory Note,08/13/2032,08/13/2025,94,9.00% PIK,investments at fair value,NaN,129,94,-35
79,"Vivos Holdings, LLC",Consumer Products,Warrants,NaN,08/13/2025,592,NaN,investments at fair value,*,0,81,81
80,"W&T Offshore, Inc.",Oil & Gas Exploration & Production,"2nd Lien, Secured Bond",02/02/2029,01/14/2025,1900,10.75%,investments at fair value,NaN,1632,1945,313
81,"Walor North America, Inc",Industrial,"1st Lien, Secured Loan",06/17/2028,06/17/2025,1369,1M SOFR + 5.75% (9.55%),investments at fair value,NaN,1369,1363,-6


### Dumping to GS

In [133]:
SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive"
]

creds = Credentials.from_service_account_file(
    "service_account.json",
    scopes=SCOPES
)

client = gspread.authorize(creds)

In [134]:
df_map = {
    "Summary": df_summary,
    "All Investments": df_investment_formatted,
    "Instrument Level Analysis": df_instrument_level_analysis,
    "Industry Level Analysis": df_industry_level_analysis,
    "Compnay Level Analysis": df_company_level_analysis,
    "Investment Type level Analysis": df_investmentsType_level_analysis
}

In [135]:
spreadsheet = client.open("GECC-10Q-20260331")

In [136]:
for tab_name, df_analysis in df_map.items():
    try:
        worksheet = spreadsheet.worksheet(tab_name)
        worksheet.clear()
    except gspread.WorksheetNotFound:
        worksheet = spreadsheet.add_worksheet(
            title=tab_name,
            rows=max(len(df_analysis) + 10, 100),
            cols=max(len(df_analysis.columns) + 10, 20)
        )

    display(df_analysis)
    set_with_dataframe(worksheet, df_analysis)
    
    header_fmt = CellFormat(
        backgroundColor=Color(0.85, 0.90, 1.0),
        textFormat=TextFormat(bold=True)
    )

    format_cell_range(
        worksheet,
        "1:1",
        header_fmt
    )

    worksheet.freeze(rows=1)
    worksheet.columns_auto_resize(0, len(df_analysis.columns))

,a,b
0,Total Investment Count,83
1,Total Cost,348231
2,Total FV,276763
3,P&L,-71468


,Portfolio Company,Industry,Security,Maturity Date,Initial Acquisition Date,Par Amount / Quantity,Interest Rate,Investment Type,Percentage of Class,Cost,Fair Value,P&L
0,"ACTIV8 Health, LLC",Insurance,"1st Lien, Secured Loan",02/03/2031,02/03/2026,3696,3M SOFR + 5.75% (9.41%),investments at fair value,NaN,3660,3655,-5
1,Advancion,Chemicals,"1st Lien, Secured Loan",11/24/2027,08/26/2025,2974,3M SOFR + 4.00% (7.77%),investments at fair value,NaN,2944,2837,-107
2,American Coastal Insurance Corp.,Insurance,Unsecured Bond,12/15/2027,12/20/2022,8000,7.25%,investments at fair value,NaN,6046,7945,1899
3,Auction.com,Financial Services,"1st Lien, Secured Loan",05/26/2028,09/09/2024,1995,6M SOFR + 6.00% (9.63%),investments at fair value,NaN,1921,1661,-260
4,"CLO Formation JV, LLC",Structured Finance,Common Equity,NaN,04/23/2024,166,NaN,investments at fair value,71.25%,52358,37590,-14768
...,...,...,...,...,...,...,...,...,...,...,...,...
78,"Vivos Holdings, LLC",Consumer Products,Promissory Note,08/13/2032,08/13/2025,94,9.00% PIK,investments at fair value,NaN,129,94,-35
79,"Vivos Holdings, LLC",Consumer Products,Warrants,NaN,08/13/2025,592,NaN,investments at fair value,*,0,81,81
80,"W&T Offshore, Inc.",Oil & Gas Exploration & Production,"2nd Lien, Secured Bond",02/02/2029,01/14/2025,1900,10.75%,investments at fair value,NaN,1632,1945,313
81,"Walor North America, Inc",Industrial,"1st Lien, Secured Loan",06/17/2028,06/17/2025,1369,1M SOFR + 5.75% (9.55%),investments at fair value,NaN,1369,1363,-6


,Security,Total Investments Count,Total Security Cost,Total Security FV,Total Security P&L,Profitable Security Count,Profitable %,Breakeven Security Count,Breakeven %,Loss Security Count,Loss %
1,"1st Lien, Secured Loan",47,146287,127410,-18877,6,0.13,3,0.06,38,0.81
5,Common Equity,8,93419,56161,-37258,0,0.0,2,0.25,6,0.75
12,Subordinated Note,1,25325,25325,0,0,0.0,1,1.0,0,0.0
3,"2nd Lien, Secured Loan",4,31878,11808,-20070,2,0.5,0,0.0,2,0.5
15,Unsecured Bond,3,9567,11374,1807,1,0.33,0,0.0,2,0.67
7,Money Market,1,9600,9600,0,0,0.0,1,1.0,0,0.0
0,"1st Lien, Secured Bond",3,7847,7796,-51,1,0.33,0,0.0,2,0.67
8,Preference Shares,1,5000,6975,1975,1,1.0,0,0.0,0,0.0
9,Private Fund,2,4311,6496,2185,2,1.0,0,0.0,0,0.0
4,CLO Equity,1,6190,6482,292,1,1.0,0,0.0,0,0.0


,Industry,Total Investment Count,Total Investment Cost,Total Investment FV,Total Investment P&L,Profitable Investment Count,Profitable %,Breakeven Investment Count,Breakeven %,Loss Investment Count,Loss %
25,Structured Finance,2,58548,44072,-14476,1,0.5,0,0.0,1,0.5
24,Specialty Finance,2,42325,37790,-4535,0,0.0,1,0.5,1,0.5
26,Technology,8,27981,30034,2053,5,0.62,0,0.0,3,0.38
4,Consumer Products,9,26218,26373,155,3,0.33,1,0.11,5,0.56
14,Insurance,3,14706,18575,3869,2,0.67,0,0.0,1,0.33
13,Industrial,4,13989,13522,-467,0,0.0,0,0.0,4,1.0
11,Food & Staples,8,29785,13289,-16496,1,0.12,1,0.12,6,0.75
3,Chemicals,6,20968,11370,-9598,1,0.17,2,0.33,3,0.5
23,Short-Term Investments,1,9600,9600,0,0,0.0,1,1.0,0,0.0
18,Metals & Mining,6,14142,8392,-5750,1,0.17,0,0.0,5,0.83


,Portfolio Company,Total Investments Count,Total Company Cost,Total Company FV,Total Company P&L,Profitable Company Count,Profitable %,Breakeven Company Count,Breakeven %,Loss Company Count,Loss %
25,"Great Elm Specialty Finance, LLC",2,42325,37790,-4535,0,0.0,1,0.5,1,0.5
4,"CLO Formation JV, LLC",1,52358,37590,-14768,0,0.0,0,0.0,1,1.0
56,"Vivos Holdings, LLC",5,17253,17398,145,2,0.4,1,0.2,2,0.4
10,"Coreweave Compute Acquisition Co. II, LLC",1,10912,11157,245,1,1.0,0,0.0,0,0.0
31,MFB Northern Inst Funds Treas Portfolio Premie...,1,9600,9600,0,0,0.0,1,1.0,0,0.0
52,Universal Fiber Systems,4,12829,8168,-4661,1,0.25,2,0.5,1,0.25
2,American Coastal Insurance Corp.,1,6046,7945,1899,1,1.0,0,0.0,0,0.0
36,"NY Daily News Enterprises, LLC",1,7000,7000,0,0,0.0,1,1.0,0,0.0
50,Trouvaille Re Ltd.,1,5000,6975,1975,1,1.0,0,0.0,0,0.0
14,Dorel Industries Inc.,2,6555,6592,37,1,0.5,0,0.0,1,0.5


,Investment Type,Total Investment Count,Total Investment Type Cost,Total Investment Type FV,Total Investment Type P&L,Profitable Investment Type Count,Profitable %,Breakeven Investment Type Count,Breakeven %,Loss Investment Type Count,Loss %
0,investments at fair value,82,338631,267163,-71468,21,0.26,8,0.1,53,0.65
1,short-term investments,1,9600,9600,0,0,0.0,1,1.0,0,0.0


In [131]:
for i in df_investment['investment_type'].unique(): print(i)

investments at fair value
short-term investments
